# MindFlow Step 4a · Add information to the picture, step by step

**Question.** The image CNN trails HGB on volatility. Which missing information explains the gap, and can the image model reach HGB when it gets it? Is the image still useful once the numbers are in?


**Diagnosis.** Every chart is scaled to its own low/high and its own maximum volume, so a quiet and a wild stock draw
almost the same picture: the *level* of volatility is not in the image, while HGB gets `realized_vol` as a number.

**The ladder** (all on the same canvas, the same CNN, the same 300,000 training windows and identical validation rows):

| rung | model | what is added |
|---|---|---|
| 0 | `plain` | the original chart (control: should match Step 3a's image CNN) |
| 1 | image CNN + SPY panel | the market's chart below the stock's (from Step 3a, joined on the same rows) |
| 2 | `ewsd` | an exponentially weighted volatility band under the chart on **one fixed log scale** (fitted on train) |
| 3 | `ma_ewsd` | + a moving average of the previous session drawn over the bars |
| 4 | `ma_ewsd_tab` | + the 38 hand + SPY numbers HGB uses, through a small MLP whose logit is added to the CNN's |
| control | `tab_only` | the same MLP on the 38 numbers, **no image**: is the image still useful? |
| control | `plain_tab` | the original chart + the 38 numbers |
| benchmark | HGB hand + SPY | from Step 3a |

**Image validity test.** If `ma_ewsd_tab` (or `plain_tab`) is significantly above `tab_only` in a paired day bootstrap,
the picture carries information that the 38 numbers do not. The feature-only MLP is the fairest control because it is
exactly the feature branch of the hybrid. Step 4b then asks which of the 38 numbers matter.


<details><summary>Where this notebook sits in the MindFlow</summary>


| step | question | notebooks |
|---|---|---|
| **1** | Does the JKX chart signal exist at the daily frequency on our large caps (also for pretrained models)? | `Step1a_Daily_JKX_CNN`, `Step1b_Daily_Pretrained` |
| **2** | Moved to 5-minute charts: does any model (CNN, ResNet, 1D CNN, ViT, HGB, frozen DINOv2/CLIP, time-series foundation models) find direction? | `Step2a_Intraday_Direction_Models`, `Step2b_Intraday_Direction_Pretrained` |
| **3** | Same models, target = volatility: is there a signal, and who wins? | `Step3a_Intraday_Volatility_Models`, `Step3b_Intraday_Volatility_Pretrained` |
| **4** | Add information to the picture step by step until the image model reaches HGB; is the image still useful beyond the features; which features matter? | `Step4a_Add_Image_Information`, `Step4b_Feature_Selection` |
| **5** | One figure and one parameter table over all steps | `Step5_Summary` |
| **6** | Next questions: does market-regime information (VIX, multi-day realized volatility) help? | `Step6a_VIX_Regime`, `Step6b_Final_Model` (final model + held-out test), `Step6c_Volatility_Use` (IC, long-short, risk sizing), `Step6d_Pure_Image` (the numbers drawn into the picture, no numbers input), `Step6f_Compact_Images` (the same picture with less black: quarter height, strips as channels, filled), `Step6g_Regularization` (weight EMA, weight decay, dropout, label smoothing, seed ensemble on the pure-image CNN) |
| **7** | Make the model small and fast: knowledge distillation | `Step7b_Distillation` (small students taught by the final CNN), `Step7b_plus_Pure_Image_Distillation` (small pure-image students taught by the 6d pure-image CNN) |
| **8** | What did the pure-image CNN learn? (occlusion, activation x weight, saliency, probing; no training) | `Step8a_Inside_Pure_CNN` |

</details>

**Data.** `alldata_bundle_v4.zip` (TAQ filter v4, point-in-time price adjustment): S&P 500 ∪ NASDAQ 100 ∪ DJIA 30
members by date, 2,514 sessions 2016-01-04 … 2025-12-31, five-minute bars plus SPY on the same clock.
**Split.** Train 2016–2022 · validation 2023 (every choice) · test 2024–2025 (last section, run once at the end).
**Advisor rule.** Every model prints its parameter count; for HGB the count is tree nodes.

## 1. Setup
Put `alldata_bundle_v4.zip` and the `code/` folder in `Demo/MindFlow` on Drive. The bundle (1.3 GB) is copied to the
Colab disk and unzipped once per runtime; the next notebook in the same runtime reuses it.

In [ ]:
import os, sys, json, shutil, time
from pathlib import Path

LOCAL_TEST = os.environ.get("MINDFLOW_LOCAL_TEST") == "1"          # repository QA only (synthetic data)
if LOCAL_TEST:
    HERE, LOCAL_ROOT = Path(os.environ["MINDFLOW_HERE"]), Path(os.environ["MINDFLOW_LOCAL"])
else:
    from google.colab import drive
    drive.mount("/content/drive")
    HERE, LOCAL_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/EC/Project/Demo/MindFlow"), Path("/content/mindflow")
CODE = HERE / "code"
assert (CODE / "mindflow.py").is_file(), f"Put the code/ folder in {HERE}"
sys.path[:0] = [str(CODE)]
import mindflow as mf
PREPARED = mf.stage_bundle(HERE / mf.BUNDLE_NAME, LOCAL_ROOT)
PANELS_ADDED = mf.stage_panels(HERE, PREPARED)          # e.g. vixy_panel_v4.zip, if uploaded
import numpy as np, pandas as pd, torch
from IPython.display import display, Markdown
import chartcnn
import indicator_cnn as ic
assert chartcnn.__version__ >= "2026-11-23.1", f"stale chartcnn {chartcnn.__version__} in {CODE}"
print("indicator_cnn", ic.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name() if DEVICE.type == "cuda" else "")

## 2. Configuration
**To resume**, set `RUN_ID`. `REFERENCE_RUN = None` uses the newest Step 3a run (vol cells at the same horizons).
`QUICK = True`: 40,000 training windows, 3 epochs, every 5th validation day, 5 minutes only (~25 min for 7 variants;
prints images per second so you can judge the full run).

**Full run (estimate on an L4):** about 20–35 min per CNN variant per cell, `tab_only` a few minutes, so
roughly 3–4 hours per cell and 6–8 hours for 5 and 60 minutes. Reruns resume per variant.

In [ ]:
SMOKE = LOCAL_TEST
assert DEVICE.type == "cuda" or SMOKE, "Select a GPU runtime"
RUN_ID = None
REFERENCE_RUN = None                               # None = newest Step 3a run
QUICK = False
VARIANTS = ["plain", "ewsd", "ma_ewsd", "ma_ewsd_tab", "tab_only", "plain_tab", "ma"]   # run order: the ladder first
HORIZONS = [1, 12]                                 # bars of 5 minutes
KINDS = ["vol"]
SETTINGS = dict(
    stride=12, embargo_sessions=2, max_train_windows=300_000, validation_every_kth_day=1, early_stop_every_kth_day=4,
    halflife=39, learning_rate=1e-5, weight_decay=0.0, batch_size=128, eval_batch_size=1024, epochs=8, patience=2,
    min_delta=1e-5, channels=[64, 128, 256], head_height=4, dropout=0.5, amp=True,
    tab_hidden=64, tab_learning_rate=1e-3,
    num_workers=0 if SMOKE else min(12, os.cpu_count() or 2), seed=69,
)
if QUICK:
    SETTINGS.update(max_train_windows=40_000, validation_every_kth_day=5, epochs=3); HORIZONS = [1]
if SMOKE:
    SETTINGS.update(max_train_windows=400, batch_size=32, eval_batch_size=64, epochs=2, early_stop_every_kth_day=1,
                    validation_every_kth_day=8)
    HORIZONS = [1]
CELLS = [dict(name=f"{k}_h{h}", kind=k, target=ic.KINDS[k], horizon=h, minutes=5 * h, stride=SETTINGS["stride"])
         for h in HORIZONS for k in KINDS]
RUN_ID, RUN_DRIVE, RESUME = mf.new_run(HERE, "4a", RUN_ID, "indicator", LOCAL_TEST)
RUN_LOCAL = LOCAL_ROOT / "work" / RUN_DRIVE.parent.name / RUN_ID
RUN_LOCAL.mkdir(parents=True, exist_ok=True)
REF_DIR = mf.latest_run(HERE, "3a", REFERENCE_RUN)
mf.save_settings(RUN_DRIVE, RESUME, settings=SETTINGS, variants=VARIANTS, cells=CELLS, quick=QUICK, reference=str(REF_DIR),
                 chartcnn=chartcnn.__version__, indicator_cnn=ic.__version__)
print("RUN_ID:", RUN_ID, "->", RUN_DRIVE)
print("Step 3a reference run:", REF_DIR)
print(len(CELLS), "cells:", ", ".join(c["name"] for c in CELLS), "|", len(VARIANTS), "variants:", ", ".join(VARIANTS))

## 3. Windows, purge and embargo
An input spans two sessions (the previous one feeds MA and EWSD), so two sessions are removed on each side of a boundary.

In [ ]:
store = chartcnn.Store(PREPARED)
cfg = store.config
EMBARGO = ic.embargo_dates(store, SETTINGS["embargo_sessions"])
W = {}
for split, kw in (("train", dict(max_rows=SETTINGS["max_train_windows"])),
                  ("valid", dict(every_kth_day=SETTINGS["validation_every_kth_day"]))):
    path = RUN_DRIVE / f"windows_{split}.parquet"
    if path.exists():
        W[split] = pd.read_parquet(path)
    else:
        W[split] = ic.decision_windows(store, "train" if split == "train" else "validation", SETTINGS["stride"],
                                       EMBARGO, seed=SETTINGS["seed"], **kw)
        W[split].to_parquet(path, index=False)
first_valid_input = store.dates[W["valid"].end_index.min() - ic.LOOKBACK - ic.HISTORY + 1]
last_train_label = store.dates[W["train"].end_index.max() + max(HORIZONS)]
assert first_valid_input > cfg["train_end"], "a validation input reaches into the training period"
assert last_train_label <= cfg["train_end"], "a training label ends after the training period"
print(f"train windows {len(W['train']):,}; validation windows {len(W['valid']):,} on {W['valid'].decision_date.nunique()} days")

## 4. Preprocessing fitted on train only, what the CNN sees, parameter counts

In [ ]:
import matplotlib.pyplot as plt
SCALE_PATH = RUN_DRIVE / "preprocessing.json"
if SCALE_PATH.exists():
    SCALE = tuple(json.loads(SCALE_PATH.read_text())["ewsd_log_scale"])
else:
    SCALE = ic.fit_ewsd_scale(PREPARED, W["train"], SETTINGS["halflife"], n=500 if SMOKE else 20_000, seed=SETTINGS["seed"])
TAB_RAW = {s: ic.tab_features(store, W[s]) for s in W}
TAB_SCALER = ic.TabScaler(TAB_RAW["train"])                      # train rows only
TAB = {s: TAB_SCALER(TAB_RAW[s]) for s in W}
if not SCALE_PATH.exists():
    SCALE_PATH.write_text(json.dumps(dict(ewsd_log_scale=SCALE, halflife=SETTINGS["halflife"], tab=TAB_SCALER.state())))
print(f"EWSD band: log scale {SCALE[0]:.2f} .. {SCALE[1]:.2f}; {TAB['train'].shape[1]} hand/SPY numbers:", ", ".join(TAB_SCALER.columns))
rv = TAB_RAW["train"]["realized_vol"].to_numpy()
pick = W["train"].iloc[[int(np.nanargmin(np.abs(rv - np.nanpercentile(rv, q)))) for q in (5, 95)]].reset_index(drop=True)
show = ["plain", "ma", "ewsd", "ma_ewsd"]
fig, axes = plt.subplots(2, len(show), figsize=(3.2 * len(show), 7.2))
for r in range(2):
    for c_, v in enumerate(show):
        img = ic.Charts(PREPARED, pick, v, SCALE, SETTINGS["halflife"]).image(r)
        axes[r, c_].imshow(img, cmap="gray", aspect="auto"); axes[r, c_].set_xticks([]); axes[r, c_].set_yticks([])
        axes[r, c_].set_title(f"{v}\n{'quiet' if r == 0 else 'volatile'}", fontsize=9)
fig.tight_layout(); fig.savefig(RUN_DRIVE / "what_the_cnn_sees.png", dpi=110); plt.show()
w0 = ic.Charts(PREPARED, pick, "plain", SCALE, SETTINGS["halflife"]).long_window(0)
assert (ic.draw_chart(w0[-ic.LOOKBACK:]) == chartcnn.images.render_window(w0[-ic.LOOKBACK:])).all()
PARAM_ROWS = []
for v in VARIANTS:
    m = ic.IndicatorCNN(SETTINGS, TAB["train"].shape[1] if ic.VARIANTS[v][2] else 0, image=v not in ic.IMAGE_FREE)
    n = ic.count(m)
    PARAM_ROWS.append(dict(variant=v, model=ic.VARIANT_LABEL[v], total=n["total"], cnn=n["cnn"], feature_mlp=n["tab"]))
PARAMS = pd.DataFrame(PARAM_ROWS).set_index("variant"); PARAMS.to_csv(RUN_DRIVE / "network_parameters.csv"); display(PARAMS)
del m

## 5. Train every variant (validation 2023)

In [ ]:
RESULTS, t0 = {}, time.perf_counter()
for i, cell in enumerate(CELLS, 1):
    print("\n" + "=" * 20, f"[{i}/{len(CELLS)}]", cell["name"], "=" * 20)
    tr, median = ic.cell_rows(store, cell, "train", W["train"])
    va, _ = ic.cell_rows(store, cell, "validation", W["valid"], median)
    assert (store.dates[tr.end_index + cell["horizon"]] <= cfg["train_end"]).all()          # purge
    assert (store.dates[va.end_index + cell["horizon"]] <= cfg["validation_end"]).all()
    es_days = np.sort(va.decision_date.unique())[::SETTINGS["early_stop_every_kth_day"]]
    es = va[va.decision_date.isin(es_days)].reset_index(drop=True)
    data = dict(train=tr, es=es, valid=va, tab_train=TAB["train"], tab_valid=TAB["valid"], scale=SCALE,
                prior=float(tr.label.mean()))
    RESULTS[cell["name"]] = ic.run_cell(PREPARED, cell, data, VARIANTS, SETTINGS, DEVICE,
                                        RUN_LOCAL / cell["name"], RUN_DRIVE / cell["name"])
    print(f"elapsed {(time.perf_counter() - t0) / 60:.1f} min")

## 6. Compare on identical rows (Step 3a scores joined)

In [ ]:
JOINED = {}
for name, r in RESULTS.items():
    pred, share = ic.join_reference(r["predictions"], REF_DIR / name) if REF_DIR else (r["predictions"], None)
    print(f"{name}: {len(pred):,} rows" + ("  (no Step 3a run)" if share is None else f", {share:.1%} matched in Step 3a"))
    JOINED[name] = dict(info=r["info"], predictions=pred)
TABLE, BOOTS = ic.table(JOINED, n_boot=50 if SMOKE else 300)
TABLE.to_csv(RUN_DRIVE / "indicator_table.csv")
MAIN = [ic.pretty("ind_" + v) for v in VARIANTS] + ["image CNN (ref run)", "image CNN + SPY panel (ref run)", "HGB hand + SPY (ref)"]
for m in MAIN:
    if m in set(TABLE.index.get_level_values("model")):
        print(m); display(ic.auc_matrix(TABLE, m))
ic.plot(TABLE, MAIN, RUN_LOCAL / "indicator_auc_by_horizon.png")

## 7. The ladder and the image-validity test
Each rung's AUC with its 95% interval; the dashed line is HGB hand + SPY. Paired differences (same resampled days):
"better"/"worse" when the 95% interval of the difference excludes zero.

In [ ]:
LADDER = [("ind_plain", "0  original chart"), ("cnn_plain_spy", "1  + SPY panel (Step 3a)"),
          ("ind_ewsd", "2  + EW volatility band"), ("ind_ma_ewsd", "3  + moving average"),
          ("ind_ma_ewsd_tab", "4  + 38 hand/SPY numbers"), ("ind_tab_only", "control: numbers only, no image"),
          ("hgb_hand_spy", "benchmark: HGB hand + SPY")]
t = TABLE.reset_index()
rows = []
for cell in RESULTS:
    for raw, label in LADDER:
        r = t[(t.cell == cell) & (t.raw == raw)]
        if len(r):
            r = r.iloc[0]
            rows.append(dict(cell=cell, rung=label, raw=raw, auc=r.roc_auc, low=r.auc_ci_low, high=r.auc_ci_high))
LADDER_TABLE = pd.DataFrame(rows)
LADDER_TABLE.to_csv(RUN_DRIVE / "ladder.csv", index=False)
display(LADDER_TABLE.pivot(index="rung", columns="cell", values="auc").round(4))
fig, axes = plt.subplots(1, len(RESULTS), figsize=(6.5 * len(RESULTS), 4.2), squeeze=False)
for ax, cell in zip(axes[0], RESULTS):
    d = LADDER_TABLE[(LADDER_TABLE.cell == cell) & ~LADDER_TABLE.raw.eq("hgb_hand_spy")].reset_index(drop=True)
    y = np.arange(len(d))[::-1]
    ax.errorbar(d.auc, y, xerr=[d.auc - d.low, d.high - d.auc], fmt="o", capsize=3)
    ax.set_yticks(y, d.rung, fontsize=8)
    h = LADDER_TABLE[(LADDER_TABLE.cell == cell) & LADDER_TABLE.raw.eq("hgb_hand_spy")]
    if len(h):
        ax.axvline(h.auc.iloc[0], ls="--", c="tab:orange", label=f"HGB hand + SPY {h.auc.iloc[0]:.3f}")
        ax.axvspan(h.low.iloc[0], h.high.iloc[0], color="tab:orange", alpha=0.12)
        ax.legend(fontsize=8, loc="lower right")
    ax.set(title=f"{cell}: validation AUC", xlabel="AUC")
fig.tight_layout(); fig.savefig(RUN_DRIVE / "ladder.png", dpi=130); plt.show()
for a, b, label in ic.CHECKS:
    p = ic.paired(BOOTS, a, b)
    if len(p):
        print(label, f"({ic.pretty(a)} − {ic.pretty(b)})"); display(p.round(4))
TAKEAWAYS = ic.takeaways(TABLE, BOOTS)
(RUN_DRIVE / "takeaways.md").write_text(TAKEAWAYS + "\n"); display(Markdown(TAKEAWAYS))

### How to read it
- **Rung 2 > rung 0**: the picture was missing the volatility level; a fixed-scale band gives it to the CNN.
- **Rung 4 tied with HGB**: with the same information the image model reaches HGB.
- **Rung 4 > `tab_only` (paired, excluding zero)**: the image adds something the 38 numbers do not. **Tied**: the
  hybrid's gain comes from the numbers; the image is redundant once they are in.
- **`tab_only` < HGB**: the MLP uses the same 38 numbers less well than the trees (a fair caveat for the test above).

## 8. Full results table

In [ ]:
display(TABLE.drop(columns=["raw"]).round(4))